# 03. Word TF-IDF baseline

Owner: rdevasia25. Issue #3. Depends on merged #2.


## Goal

Fit one word model on the fixed 8,000 training rows. Measure it on the fixed 2,000 validation rows and save a checked local handoff.


## Setup

TF-IDF gives less weight to words seen in many training passages. Logistic regression turns a weighted feature sum into P(AI).


In [ ]:
from pathlib import Path
import json, time, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

# Notebook 02 defines helpers only. No fits or downloads on import.
get_ipython().run_line_magic("run", "./02_split_metrics.ipynb")
NAME = "03_word_baseline"
COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace notebooks/03_word_baseline.ipynb"
CONFIG = {
    "vectorizer": {"analyzer": "word", "ngram_range": [1, 2], "lowercase": True,
                   "min_df": 2, "max_features": 50000, "sublinear_tf": True,
                   "norm": "l2", "dtype": "float32", "stop_words": None,
                   "token_pattern": r"(?u)\b\w\w+\b", "strip_accents": None},
    "classifier": {"C": 1.0, "solver": "liblinear", "max_iter": 2000,
                   "random_state": 42, "penalty": "l2", "tol": 1e-4,
                   "fit_intercept": True, "class_weight": None,
                   "intercept_scaling": 1.0},
    "threads": 1, "prediction_tolerance": 1e-12,
}
notebook = json.loads((ROOT / "notebooks/03_word_baseline.ipynb").read_text())
SOURCE_HASH = value_hash(["".join(cell["source"]) for cell in notebook["cells"]])
RUN_CONFIG = dict(CONFIG, notebook_source_sha256=SOURCE_HASH)
START = time.perf_counter()


## Checks

Load rows in manifest order and verify raw bytes, audit identity, and duplicate exclusions. Stop if private inputs are missing. IDs align outputs; only text enters the model.


In [ ]:
try:
    parts = load_partitions()
except (OSError, ValueError) as exc:
    # The shared recorder requires valid data. Record preflight failure separately.
    split = json.loads(SPLIT_PATH.read_text())
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    failure_path = ROOT / "results" / NAME / f"{run_id}-preflight-failed.json"
    save_new_json(failure_path, {
        "run_id": run_id, "status": "failed", "stage": "input_preflight",
        "error_type": type(exc).__name__, "utc": datetime.now(timezone.utc).isoformat(),
        "command": COMMAND,
        "git_sha": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        "tracked_worktree_dirty": bool(subprocess.check_output(
            ["git", "diff", "HEAD", "--name-only"], cwd=ROOT, text=True).strip()),
        "versions": {**{name: importlib.metadata.version(name) for name in
                     ["numpy", "pandas", "scipy", "scikit-learn", "joblib", "matplotlib", "nbconvert", "ipykernel"]},
                     "python": platform.python_version()},
        "lock_sha256": file_hash(ROOT / "uv.lock"), "seed": SEED,
        "config": RUN_CONFIG, "config_sha256": value_hash(RUN_CONFIG),
        "expected_data_sha256": split["data_sha256"], "inputs_verified": False,
        "split_sha256": split["split_sha256"], "split_file_sha256": file_hash(SPLIT_PATH),
        "fit_rows": 0, "eval_rows": 0, "metrics": {"validation_auroc": None},
        "runtime_seconds": time.perf_counter() - START, "artifacts": {},
    })
    raise RuntimeError("Input preflight failed. Restore matching private data and audit artifacts; see safe failure record.") from None
train, validation = parts["train"], parts["validation"]
require(len(train) == 8000 and len(validation) == 2000, "Wrong split sizes")
require(train.id.tolist() == parts["split"]["train_ids"], "Training order changed")
require(validation.id.tolist() == parts["split"]["validation_ids"], "Validation order changed")
for frame in [train, validation]:
    require(frame.text.notna().all() and frame.text.map(lambda s: isinstance(s, str) and bool(s.strip())).all(),
            "Expected nonempty text")
print("Verified fixed 8000/2000 split. Only training text will be fitted.")


## Method

Lowercase word unigrams and bigrams, with sublinear TF and L2 normalization. Keep original passages unchanged, with no stemming or stopword removal. The pipeline learns vocabulary, IDF, and classifier weights from training only.


In [ ]:
def make_baseline():
    vector_settings = dict(CONFIG["vectorizer"])
    vector_settings["ngram_range"] = tuple(vector_settings["ngram_range"])
    vector_settings["dtype"] = np.float32
    return Pipeline([
        ("tfidf", TfidfVectorizer(**vector_settings)),
        ("logistic", LogisticRegression(**CONFIG["classifier"])),
    ])

def prediction_frame(model, frame):
    return pd.DataFrame({"id": frame.id.to_numpy(),
                         "p_ai": class1_probabilities(model, frame.text)})


## Run

Fit once with seed 42 and one numerical thread. A convergence warning fails the run so an unfinished fit cannot pass quietly.


In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT / "artifacts" / NAME / run_id
private.mkdir(parents=True, exist_ok=False)
try:
    model = make_baseline()
    with threadpool_limits(limits=CONFIG["threads"]), warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        fit_start = time.perf_counter()
        model.fit(train.text, train.label)
        fit_seconds = time.perf_counter() - fit_start
        eval_start = time.perf_counter()
        train_predictions = prediction_frame(model, train)
        validation_predictions = prediction_frame(model, validation)
        train_metrics = evaluate_predictions(train, train_predictions)
        validation_metrics = evaluate_predictions(validation, validation_predictions)
        eval_seconds = time.perf_counter() - eval_start
    model_path = private / "model.joblib"
    joblib.dump(model, model_path)
    train_path, validation_path = private / "train_predictions.csv", private / "validation_predictions.csv"
    train_predictions.to_csv(train_path, index=False)
    validation_predictions.to_csv(validation_path, index=False)
    reload_start = time.perf_counter()
    restored = joblib.load(model_path)
    with threadpool_limits(limits=CONFIG["threads"]):
        reloaded = prediction_frame(restored, validation)
    max_reload_difference = float(np.max(np.abs(reloaded.p_ai.to_numpy() - validation_predictions.p_ai.to_numpy())))
    require(max_reload_difference <= CONFIG["prediction_tolerance"], "Reload predictions changed")
    for frame, path in [(train, train_path), (validation, validation_path)]:
        saved = pd.read_csv(path, dtype={"id": "string"}, float_precision="round_trip")
        require(saved.id.tolist() == frame.id.tolist(), "Saved prediction order changed")
        require(evaluate_predictions(frame, saved) == evaluate_predictions(frame, prediction_frame(restored, frame)),
                "Saved prediction metrics changed")
    reload_seconds = time.perf_counter() - reload_start
except Exception as exc:
    record_run(NAME, COMMAND, RUN_CONFIG, SEED, len(train), len(validation),
               {"validation_auroc": None}, time.perf_counter() - START,
               status="failed", error_type=type(exc).__name__)
    raise RuntimeError("Baseline failed. See safe run record; inspect private inputs locally.") from None


## Results

Training AUROC describes fit to seen labels. Validation AUROC measures this held-out split; it is not a Kaggle score. This baseline has one fixed candidate, with no search or discarded arms.


In [ ]:
summary = pd.DataFrame([
    {"partition": "train", **train_metrics},
    {"partition": "validation", **validation_metrics},
])
display(summary)
print(f"Fit: {fit_seconds:.3f}s; evaluation: {eval_seconds:.3f}s; reload checks: {reload_seconds:.3f}s")
print(f"Vocabulary: {len(model.named_steps['tfidf'].vocabulary_)}; iterations: {model.named_steps['logistic'].n_iter_.tolist()}")
print(f"Maximum reload probability difference: {max_reload_difference:.3g}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(summary.partition, summary.auroc)
ax.set(ylim=(0, 1), ylabel="AUROC", title="Fixed word baseline")
fig.tight_layout()
plot_path = private / "auroc.png"
fig.savefig(plot_path, dpi=140)
plt.show()


## Takeaway

- Tried one fixed TF-IDF logistic model. No tuning here.
- The table shows measured fit and held-out performance. Reload checks protect the handoff.
- Next: compare word learners in #4. Keep weaker models too.


## Save

Each run keeps its own model and ID-aligned predictions under ignored artifacts. `artifacts/03_word_baseline/latest.json` points downstream notebooks to the latest successful run and its safe aggregate record. No passage or row prediction goes into Git.


In [ ]:
metrics = {
    "train_auroc": train_metrics["auroc"], "validation_auroc": validation_metrics["auroc"],
    "train_accuracy_at_0_5": train_metrics["accuracy_at_0_5"],
    "validation_accuracy_at_0_5": validation_metrics["accuracy_at_0_5"],
    "fit_seconds": fit_seconds, "evaluation_seconds": eval_seconds,
    "reload_check_seconds": reload_seconds, "max_reload_probability_difference": max_reload_difference,
    "vocabulary_size": len(model.named_steps["tfidf"].vocabulary_),
    "iterations": model.named_steps["logistic"].n_iter_.tolist(),
}
RUN_CONFIG["extra_versions"] = {name: importlib.metadata.version(name) for name in
                                ["joblib", "matplotlib", "threadpoolctl"]}
record_path = record_run(NAME, COMMAND, RUN_CONFIG, SEED, len(train), len(validation),
                         metrics, time.perf_counter() - START,
                         artifacts=[model_path, train_path, validation_path, plot_path])
# Publish the pointer only after all checks and the aggregate record succeed.
pointer = {"result": str(record_path.relative_to(ROOT)),
           "result_sha256": file_hash(record_path),
           "model": str(model_path.relative_to(ROOT)),
           "train_predictions": str(train_path.relative_to(ROOT)),
           "validation_predictions": str(validation_path.relative_to(ROOT)),
           "split_sha256": parts["split"]["split_sha256"],
           "artifacts": {str(p.relative_to(ROOT)): file_hash(p) for p in [model_path, train_path, validation_path]},
           "class1_interface": "class1_probabilities(joblib.load(model), texts)"}
pointer_path = ROOT / "artifacts" / NAME / "latest.json"
temporary = pointer_path.with_suffix(".tmp")
temporary.write_text(json.dumps(pointer, indent=2, sort_keys=True) + "\n")
temporary.replace(pointer_path)
print("Saved aggregate record and checked private handoff.")
